# Muscat Furniture — free 3D models from product photos

Turns every furniture product photo into a real 3D model (GLB) with **TripoSR** (free, MIT licence) on Google Colab's free GPU.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.
Then run the cells **one by one** (▶ on each, or `Runtime → Run all`).

You need the file `models-jobs.json` from your project (`npm run models:list` creates it in `atelier/data/`).
At the end a zip downloads — import it with `npm run models:import -- <zip>`.


## 1. Check the free GPU is on

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime > Change runtime type > T4 GPU"


## 2. Install TripoSR (takes 1–3 minutes)
Red "dependency resolver" warnings are normal. At the end the session restarts by itself — that is expected.

In [ ]:
%cd /content
![ -d /content/TripoSR ] || git clone -q https://github.com/VAST-AI-Research/TripoSR.git
%cd /content/TripoSR
# no pinned versions: Colab now runs Python 3.13, and the old pins in TripoSR's requirements have no 3.13 builds
# transformers 5 renamed the image-encoder layers, so TripoSR's weights only load with version 4
!pip install -q omegaconf einops "transformers<5" trimesh huggingface-hub "rembg[cpu]" requests scikit-image
# torchmcubes no longer compiles on Python 3.13, so TripoSR gets a drop-in replacement built on scikit-image
open("/content/TripoSR/torchmcubes.py", "w").write('''
import numpy as np, torch
from skimage.measure import marching_cubes as _mc

def marching_cubes(vol, thresh=0.0):
    verts, faces, _, _ = _mc(vol.detach().float().cpu().numpy(), level=thresh)
    verts = verts[:, ::-1].copy()  # torchmcubes returns (x, y, z) = (axis 2, 1, 0); TripoSR swaps them back
    faces = faces[:, ::-1].copy()  # reversing the axes mirrors the mesh, so flip the winding to match
    return torch.from_numpy(verts).to(vol.device), torch.from_numpy(faces.astype(np.int64)).to(vol.device)
''')
print("✅ installed. The session now restarts by itself so the new packages load — wait 10 seconds, then run step 2b.")
import os
os.kill(os.getpid(), 9)  # restart: packages that were already loaded (numpy) must be reloaded from the new install


## 2b. Check the install (run after the restart)

In [ ]:
import sys
sys.path.insert(0, "/content/TripoSR")
import torch, torchmcubes
from tsr.system import TSR
print("✅ ready — torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


## 3. Upload `models-jobs.json`
Click **Choose files** and pick `atelier/data/models-jobs.json`.

In [ ]:
import json
from google.colab import files
up = files.upload()
jobs = json.loads(next(iter(up.values())).decode("utf-8"))
print(len(jobs), "products to convert")


## 4. Make the 3D models (about 5–10 seconds each)
If Colab disconnects, just run this cell again — finished products are skipped.

In [ ]:
import os, io, sys, time, traceback
import numpy as np, requests, torch, trimesh, rembg
from PIL import Image
sys.path.insert(0, "/content/TripoSR")
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")
model.renderer.set_chunk_size(8192)
model.to(device)
session = rembg.new_session("u2net")  # small model: the newer default (bria, 1 GB) runs free Colab out of memory

def y_up(mesh):
    # TripoSR meshes are Z-up; the website (glTF) is Y-up — same fix as TripoSR's own demo
    mesh.apply_transform(trimesh.transformations.rotation_matrix(-np.pi / 2, [1, 0, 0]))
    mesh.apply_transform(trimesh.transformations.rotation_matrix(np.pi / 2, [0, 1, 0]))
    return mesh

if "jobs" not in globals():  # after a session restart: reuse the file uploaded in step 3
    import json, glob
    jobs = json.load(open(glob.glob("/content/**/models-jobs*.json", recursive=True)[0]))
OUT = "/content/out"
os.makedirs(OUT, exist_ok=True)
done, failed = 0, []
for n, job in enumerate(jobs, 1):
    target = f"{OUT}/{job['slug']}.glb"
    if os.path.exists(target):
        continue
    try:
        t = time.time()
        raw = Image.open(io.BytesIO(requests.get(job["image"], timeout=60).content)).convert("RGB")
        img = resize_foreground(remove_background(raw, session), 0.85)
        arr = np.array(img).astype(np.float32) / 255.0
        arr = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5   # grey background, as TripoSR expects
        img = Image.fromarray((arr * 255.0).astype(np.uint8))
        with torch.no_grad():
            codes = model([img], device=device)
        mesh = model.extract_mesh(codes, True, resolution=256)[0]   # True = keep colours
        y_up(mesh).export(target)
        del codes, mesh
        torch.cuda.empty_cache()
        done += 1
        print(f"[{n}/{len(jobs)}] ✅ {job['name']}  ({time.time() - t:.1f}s)")
    except Exception as e:
        failed.append(job["slug"])
        print(f"[{n}/{len(jobs)}] ❌ {job['name']}: {e}")
print(f"\nDone: {done} new models, {len(failed)} failed", failed)


## 5. Download all models as one zip

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/muscat-3d-models", "zip", "/content/out")
print(len(os.listdir("/content/out")), "models in", zip_path)
files.download(zip_path)


## 6. On your laptop
```bash
cd ~/Documents/up/atelier
npm run models:import -- ~/Downloads/muscat-3d-models.zip
```
Then open the room planner — pieces now appear as their real 3D models (mode **✨ AI 3D**).